# Advanced RFT: Math Reasoning

**TRAINING · ADVANCED / QUOTA-DEPENDENT**

This notebook uses preserved 100/10 OpenR1-Math-220k subsets to teach long-form mathematical RFT. It keeps five outcomes separate: grader validity, baseline quality, training completion, deployment availability, and endpoint evaluation.

> **Training costs:** Executing upload and training cells performs Azure calls and can incur charges. Local data and grader tests do not submit jobs.

## 1. Imports, paths, and execution gates

In [ ]:
import importlib.util
import json
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
demo_candidates = [cwd, cwd.parent] if cwd.name == "notebooks" else [cwd]
DEMO_ROOT = next(
    (path for path in demo_candidates if (path / "data/hashes.json").is_file()),
    None,
)
if DEMO_ROOT is None:
    raise FileNotFoundError("Run from the math cookbook or its notebooks directory")
TRACK_ROOT = DEMO_ROOT.parent
sys.path.insert(0, str(TRACK_ROOT / "src"))

from rft_common import (
    collect_new_events,
    content_addressed_name,
    create_job_once,
    env_flag,
    extract_training_metrics,
    load_jsonl,
    require_env,
    retry,
    upload_file_once,
    validate_hash_manifest,
    wait_for_job,
)

load_dotenv(DEMO_ROOT / ".env", override=False)

FOUNDRY_RUN_REMOTE_BASELINE = env_flag("FOUNDRY_RUN_REMOTE_BASELINE")
FOUNDRY_RUN_REMOTE_EVALUATION = env_flag("FOUNDRY_RUN_REMOTE_EVALUATION")
print(
    {"remote_baseline": FOUNDRY_RUN_REMOTE_BASELINE, "remote_evaluation": FOUNDRY_RUN_REMOTE_EVALUATION}
)

## 2. Verify preserved datasets and inspect the task

In [ ]:
validate_hash_manifest(DEMO_ROOT / "data/hashes.json", DEMO_ROOT)

TRAIN_PATH = DEMO_ROOT / "data/preserved/openr1-math-training-rft-100.jsonl"
VALIDATION_PATH = DEMO_ROOT / "data/preserved/openr1-math-validation-rft-10.jsonl"
training_rows = load_jsonl(TRAIN_PATH)
validation_rows = load_jsonl(VALIDATION_PATH)

assert len(training_rows) == 100
assert len(validation_rows) == 10
assert all(set(row) == {"messages", "answer"} for row in training_rows)
assert all(set(row) == {"messages", "answer"} for row in validation_rows)
assert all(len(row["messages"]) == 1 for row in training_rows + validation_rows)
assert all(row["messages"][0]["role"] == "user" for row in training_rows + validation_rows)

train_prompts = {row["messages"][0]["content"] for row in training_rows}
validation_prompts = {row["messages"][0]["content"] for row in validation_rows}
assert train_prompts.isdisjoint(validation_prompts), "Training/validation leakage detected"

print(
    {
        "training_rows": len(training_rows),
        "validation_rows": len(validation_rows),
        "first_answer": training_rows[0]["answer"],
        "split_overlap": 0,
    }
)

Each record contains a problem prompt and a reference final answer, but no assistant reasoning trace. The model explores solution paths; the grader rewards a final `\boxed{...}` answer that matches the reference.

## 3. Load, validate, and attack the boxed-answer grader

In [ ]:
validate_hash_manifest(DEMO_ROOT / "graders/hashes.json", DEMO_ROOT / "graders")

grader_path = DEMO_ROOT / "graders/boxed_math_grader.py"
grader_spec = importlib.util.spec_from_file_location("boxed_math_grader", grader_path)
grader_module = importlib.util.module_from_spec(grader_spec)
grader_spec.loader.exec_module(grader_module)
GRADER_SOURCE = grader_path.read_text(encoding="utf-8")
GRADER = {
    "type": "python",
    "name": "core_rft_boxed_math_answer",
    "source": GRADER_SOURCE,
}

grader_tests = [
    ("exact integer", r"The final answer is \boxed{42}", "42", 1.0),
    ("wrong integer", r"The final answer is \boxed{99}", "42", 0.0),
    ("missing box", "The final answer is 42", "42", 0.0),
    ("last box wins", r"Draft \boxed{0}; final \boxed{42}", "42", 1.0),
    ("final wrong defeats earlier leak", r"Reference \boxed{42}; final \boxed{41}", "42", 0.0),
    ("nested LaTeX", r"Answer: \boxed{\sqrt{2}}", r"\sqrt{2}", 1.0),
    ("unordered answer list", r"Answer: \boxed{3, 2, 1}", "1, 2, 3", 1.0),
    ("spacing normalization", r"Answer: \boxed{x + 1}", "x+1", 1.0),
    ("unclosed box", r"Answer: \boxed{42", "42", 0.0),
    ("reward hack: answer outside final box", r"42, therefore \boxed{0}", "42", 0.0),
]

for name, output, answer, expected in grader_tests:
    actual = grader_module.grade({"output_text": output}, {"answer": answer})
    assert actual == expected, f"{name}: expected {expected}, received {actual}"
    print(f"{name:38} reward={actual}")

print(f"All {len(grader_tests)} grader and reward-hacking tests passed")

The grader intentionally judges only the last balanced `\boxed{...}` expression. Mentioning the reference elsewhere, producing an earlier correct box followed by a wrong final answer, or omitting the box earns zero reward. This limits obvious reward hacking, but it does not prove the reasoning is faithful.

## 4. Define one baseline and endpoint evaluation contract

In [ ]:
def solve_math(client, deployment, row):
    response = retry(
        lambda: client.chat.completions.create(
            model=deployment,
            messages=row["messages"],
            max_completion_tokens=20_000,
        ),
        description=f"solve math item with {deployment}",
    )
    output = response.choices[0].message.content or ""
    reward = grader_module.grade({"output_text": output}, {"answer": row["answer"]})
    return {
        "reward": reward,
        "correct": reward == 1.0,
        "word_count": len(output.split()),
        "output": output,
    }


def evaluate_math_deployment(client, deployment, rows, limit=10):
    results = [solve_math(client, deployment, row) for row in rows[:limit]]
    return {
        "deployment": deployment,
        "correct": sum(result["correct"] for result in results),
        "total": len(results),
        "mean_reward": sum(result["reward"] for result in results) / len(results),
        "results": results,
    }


baseline_result = None
if FOUNDRY_RUN_REMOTE_BASELINE:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_MATH_BASELINE_DEPLOYMENT",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    baseline_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    baseline_client = baseline_project.get_openai_client()
    baseline_result = evaluate_math_deployment(
        baseline_client,
        values["FOUNDRY_MATH_BASELINE_DEPLOYMENT"],
        validation_rows,
        limit=10,
    )
    print(
        {
            key: baseline_result[key]
            for key in ("deployment", "correct", "total", "mean_reward")
        }
    )
else:
    print("Skipped remote baseline; offline grader validation is complete.")

## 5. Prepare content-addressed files and the RFT job

The unchanged reinforcement recipe is visible offline below. Copy this module's blank `.env.template` to a module-local `.env`; executing the upload/training cell submits or resumes a paid job. Existing jobs, including pending jobs, must match the model, input file IDs, and full reinforcement recipe. A mismatch or failed/cancelled job raises an error.

Creation is attempted once with SDK retries disabled. Reconcile an ambiguous submission failure in Foundry before retrying.


In [ ]:
MATH_METHOD = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": GRADER,
        "hyperparameters": {
            "n_epochs": 2,
            "batch_size": 128,
            "learning_rate_multiplier": 2.0,
            "eval_interval": 25,
            "eval_samples": 2,
            "reasoning_effort": "medium",
        },
    },
}

remote_names = {
    "training": content_addressed_name("core-rft-openr1-math-training", TRAIN_PATH),
    "validation": content_addressed_name("core-rft-openr1-math-validation", VALIDATION_PATH),
}
remote_names

In [ ]:
fine_tuning_job = None
training_client = None
project_client = None

values = require_env("FOUNDRY_PROJECT_ENDPOINT", "FOUNDRY_BASE_MODEL")
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
    endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
training_client = project_client.get_openai_client()
training_file = upload_file_once(
    training_client,
    TRAIN_PATH,
    prefix="core-rft-openr1-math-training",
    purpose="fine-tune",
)
validation_file = upload_file_once(
    training_client,
    VALIDATION_PATH,
    prefix="core-rft-openr1-math-validation",
    purpose="fine-tune",
)

def submit_math_job():
    return training_client.with_options(max_retries=0).fine_tuning.jobs.create(
        training_file=training_file.id,
        validation_file=validation_file.id,
        model=values["FOUNDRY_BASE_MODEL"],
        method=MATH_METHOD,
        extra_body={"trainingType": "GlobalStandard"},
        suffix="core-rft-openr1-math",
    )

fine_tuning_job = create_job_once(
    training_client,
    existing_job_id=os.getenv("FOUNDRY_EXISTING_MATH_RFT_JOB_ID") or None,
    expected={
        "model": values["FOUNDRY_BASE_MODEL"],
        "training_file": training_file.id,
        "validation_file": validation_file.id,
        "method": MATH_METHOD,
    },
    create=submit_math_job,
)
print(f"Math RFT job: {fine_tuning_job.id} ({fine_tuning_job.status})")


## 6. Monitor with a timeout and collect real reward/loss events

In [ ]:
training_outcome = None
event_metrics = {"reward": [], "loss": []}

completed_job = wait_for_job(
    training_client,
    fine_tuning_job.id,
    timeout_seconds=21_600,
    poll_seconds=60,
)
seen_ids = set()
events = collect_new_events(training_client, fine_tuning_job.id, seen_ids)
event_messages = [event.message for event in events]
event_metrics = extract_training_metrics(event_messages)
training_outcome = {
    "status": completed_job.status,
    "registered_model": completed_job.fine_tuned_model,
    "eval_id": completed_job.eval,
    "event_metrics": event_metrics,
}
print(training_outcome)


![Representative validation reward](../assets/charts/representative-validation-reward.svg)

The committed curve contains the two validation means recorded by the existing run: `0.05` and `0.20`. A current run should use service events and evaluation outputs from the previous cell; never substitute this historical curve for current evidence.

## 7. Retrieve validation rewards and inspect a successful reasoning sample

In [ ]:
validation_outcome = None
completed_job = training_client.fine_tuning.jobs.retrieve(fine_tuning_job.id)
eval_id = completed_job.eval
eval_runs = list(training_client.evals.runs.list(eval_id, limit=100).data)
completed_runs = sorted(
    (run for run in eval_runs if run.status == "completed"),
    key=lambda run: run.created_at,
)
if not completed_runs:
    raise RuntimeError(f"No completed validation runs found for {eval_id}")

reward_history = []
selected_item = None
for eval_run in completed_runs:
    output_items = list(
        training_client.evals.runs.output_items.list(
            run_id=eval_run.id,
            eval_id=eval_id,
            limit=100,
        ).data
    )
    scores = [
        float(item.results[0].score)
        for item in output_items
        if item.results
    ]
    if not scores:
        raise RuntimeError(f"Evaluation run {eval_run.id} returned no scores")
    reward_history.append(sum(scores) / len(scores))
    if eval_run is completed_runs[-1]:
        selected_item = next(
            (
                item
                for item in output_items
                if item.results and float(item.results[0].score) == 1.0
            ),
            None,
        )

if selected_item is None:
    raise AssertionError("Final validation run has no correct output")
sample = selected_item.datasource_item["sample"]["output_text"]
reference = str(selected_item.datasource_item["answer"])
if grader_module.grade({"output_text": sample}, {"answer": reference}) != 1.0:
    raise AssertionError("Selected validation output does not pass the local grader")
if len(sample.split()) < 40:
    raise AssertionError("Selected output lacks a substantive reasoning trace")

validation_outcome = {
    "reward_history": reward_history,
    "correct_sample_words": len(sample.split()),
    "final_run": completed_runs[-1].id,
}
print(validation_outcome)


## 8. Check deployment availability separately

In [ ]:
deployment_outcome = None
matching = list(
    project_client.deployments.list(model_name=completed_job.fine_tuned_model)
)
if matching:
    deployment = matching[0]
    deployment_outcome = {
        "status": "available",
        "name": deployment.name,
        "model": deployment.model_name,
    }
else:
    deployment_outcome = {
        "status": "not_available",
        "model": completed_job.fine_tuned_model,
        "next_step": (
            "Confirm supported model/region and sufficient deployment quota, "
            "then create the deployment in Foundry."
        ),
    }
print(deployment_outcome)


A succeeded RFT job registers a model. Deployment availability is checked separately for your configured project.

## 9. Evaluate a deployed endpoint with the same validation contract

In [ ]:
endpoint_evaluation = None
if FOUNDRY_RUN_REMOTE_EVALUATION:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    evaluation_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    deployment = retry(
        lambda: evaluation_project.deployments.get(
            values["FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT"]
        ),
        description="retrieve math fine-tuned deployment",
    )
    evaluation_client = evaluation_project.get_openai_client()
    endpoint_evaluation = evaluate_math_deployment(
        evaluation_client,
        deployment.name,
        validation_rows,
        limit=10,
    )
    print(
        {
            key: endpoint_evaluation[key]
            for key in ("deployment", "correct", "total", "mean_reward")
        }
    )
else:
    print("Skipped endpoint evaluation; it requires an available deployment.")

## 10. Outcome distinctions, conclusion, and next step

In [ ]:
outcome_summary = {
    "grader_validation": "passed offline",
    "baseline": baseline_result or "not run",
    "training": training_outcome or "not submitted in this run",
    "validation": validation_outcome or "representative evidence shown",
    "deployment": deployment_outcome or "not checked in this run",
    "endpoint_evaluation": endpoint_evaluation or "not run",
}
outcome_summary

### Conclusion

The advanced-math run demonstrates that training completion, rising validation reward, deployment availability, and endpoint quality are different claims. The boxed-answer grader gives a useful automatic signal, but it verifies the final expression rather than the faithfulness of every reasoning step.

**Next step:** secure supported deployment capacity, add an untouched test split, evaluate the base and fine-tuned deployments on that independent set, and manually review correct and incorrect chains for reasoning quality—not just final-answer reward.

*Variability footnote:* the `0.05 → 0.20` validation rewards and 257-word correct response come from one existing run. Model availability, quota, service updates, grader behavior, data composition, and training stochasticity can materially change results.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.